In [61]:
import json
import re
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path


# from lecture 6
# create requirements.txt by using pip freeze in your repo

### Task 1

In [76]:
USER = "Bruker"
BASE_DIR = Path("C:\\", "Users", USER, "Documents", "folder_for_saving", "d2_data")

def read_file(filepath: str) -> tuple(str, str):
    """
    Opens and reads file. Checking encoding type between UTF-8 and ISO-8859-1

    ARGS: file path
    
    RETURNS: tuple containing strings for file text content and encoding.
    """

    # create a path object
    path = Path(filepath)

    # tries opening with UTF-8 encoding first and with latin-1/ISO-8859-1 second
    # This should correctly decode files encoded with UTF and ISO-8859-1
    # does not always return correct encoding as some non conflicting ISO-8859-1 files
    # can be opened in UTF-8
    try:
        filedata = open(path, encoding="UTF-8").read(), "UTF-8"
        return filedata
    except:
        filedata = open(path, encoding="ISO-8859-1").read(), "ISO-8859-1"
        return filedata


# test (not sure if the generated files from generate_data_for_d2 is the same so check generation log if encoding is correct)
# mine was. Giving data and UTF-8 encoding which is correct
# print(read_file(r"C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026\science_building\POS_02\register_sales.log")[1])

# # for me this should be ISO-8859-1. It is not. 
# print(read_file(r"C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026\science_building\POS_03\register_sales.log"))
# print(read_file(r"{BASE_DIR}\2026\science_building\POS_03\register_sales.log"))

def parse_log(filecontent: str) -> list[dict]:
    """
    Reads files line by line and checks if lines are a valid log format and then extracts
    info from that log entry.

    ARGS: Content of a register_sales.log file.

    RETURNS: A tuple list containing dictionaries where each dictionary is a valid transaction

    WARNING: Does not handle tuples from read_file
    """
    valid_sales = []
    # Set up regex pattern for the that should be matched from log
    pattern = re.compile(
        r"\[(?P<timestamp>[\d\-]+\s[\w:]+)\] "
        r"\[(?P<register_id>POS_\d+)\] "
        r"item=(?P<item>\S+) "
        r"price=(?P<price>\S+) "
        r"(?P<currency>\w+) "
        r"\((?P<status>\w+)\)$",
        re.MULTILINE
    )

    # Creates a list of dictionaries where each dictionary will be a valid transaction by
    # Iterating over the lines in filecontent  
    valid_sales = [{**match.groupdict(), "price": float(match.group("price"))} for match in pattern.finditer(filecontent)]
    return valid_sales

# Test
parse_log(read_file(r"C:\Brukere\mathe\Documents\folder_for_saving\d2_data\2026\library_cafe\POS_01\register_sales.log")[0])


[{'timestamp': '2026-10-15 07:01:35',
  'register_id': 'POS_01',
  'item': 'Muffin',
  'price': 3.59,
  'currency': 'USD',
  'status': 'CANCELLED'},
 {'timestamp': '2026-10-15 07:03:56',
  'register_id': 'POS_01',
  'item': 'Latte',
  'price': 4.15,
  'currency': 'NOK',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:04:35',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 2.88,
  'currency': 'USD',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:05:56',
  'register_id': 'POS_01',
  'item': 'Cappuccino',
  'price': 3.94,
  'currency': 'EUR',
  'status': 'CANCELLED'},
 {'timestamp': '2026-10-15 07:06:13',
  'register_id': 'POS_01',
  'item': 'Croissant',
  'price': 3.43,
  'currency': 'EUR',
  'status': 'CANCELLED'},
 {'timestamp': '2026-10-15 07:09:13',
  'register_id': 'POS_01',
  'item': 'Muffin',
  'price': 3.68,
  'currency': 'USD',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:09:55',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 2.73,


The above code should be able to sort all the given files. It will not catch files that are ambiguous which seems to be almost impossible for UTF-8 and ISO-8859-1 without external info about the file. As with the short test done above. I have not run it on the whole file set yet so I do not know if it will be able to tell a single file appart from the given set. 

### Task 2

In [78]:
def find_log_files(path: str) -> list[Path]:
    # Convert to Path object
    folder = Path(path)

    logs = []

    # iterate over items in directory
    for child in folder.iterdir():
        if child.is_file() and str(child)[-4:] == ".log":
            logs.append(child)  # if child is a .log file add Path to logs
        elif child.is_dir():
            # if child is a directory call function and add elements to logs via extend
            logs.extend(find_log_files(child))
    return logs

# Test
# print(find_log_files(r"C:\Users\Bruker\Documents\folder_for_saving\d2_data\2026"))


def get_active_registers(configpath: str) -> set[str]:
    config = Path(configpath)



### Task 3

In [ ]:
def removing_price_outliers(input_price_list: list) -> np.ndarray:
    """
    Converts a list of price values to a 1D NumPy array and replaces all hardware glitch outliers
    with the median of the valid prices.

    Args:
        input_price_list: A list of price values.
    Returns:
        cleaned_price_array: A 1D NumPy array with valid prices.
    """

    # Convert the input price data list to a 1D NumPy float array
    input_price_array = np.array(input_price_list)

    # Calculate the mean and standard deviation of the array with prices
    mean = np.mean(input_price_array)
    sigma = np.std(input_price_array)

    # Find all valid prices to later calculate the median of the valid prices
    valid_prices = []
    for price in input_price_array:
        if abs(price - mean) <= 3 * sigma:
            valid_prices.append(price)

    # Calculate the median of the valid prices
    median = np.median(valid_prices)

    # Replace all hardware glitch outliers with the median of the valid prices
    cleaned_price_array = np.where(abs(input_price_array - mean) > 3 * sigma, median, input_price_array)

    return cleaned_price_array


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Calculates the refund ratio"""

    # We have 10**(-6) in the denominator to prevent zero-devision errors
    refund_ratio = refund_count / (sales_count + 10**(-6))

    return refund_ratio


### Task 4

In [137]:
def list_to_dataframe(transaction_data: list) -> pd.DataFrame:
    """Converts a list with dictionaries to a pandas DataFrame and.
    Converts the timestamp to datetime and sets it as the index of the DataFrame."""

    # Convert the list to a DataFrame
    df_transaction_data = pd.DataFrame(transaction_data)

    # Convert the timestamp column to datetime
    df_transaction_data['timestamp'] = pd.to_datetime(df_transaction_data['timestamp'])

    # Set the datetime as index
    df_transaction_data = df_transaction_data.set_index('timestamp')

    # Sort the DataFrame chronologically by the timestamp
    df_transaction_data = df_transaction_data.sort_index()

    return df_transaction_data


In [138]:
def group_by_item(df_data: pd.DataFrame) -> pd.DataFrame:
    """Groups transaction data by item and resamples transactions to one-hour windows.
    Calculate the total prices per item per hour."""

    # Group data by item
    grouped_by_item = df_data.groupby('item')

    # Resample transactions in one-hour window and calculte the sum of the prices
    hourly_sum = grouped_by_item.resample('1h').sum()

    # Remove all columns which is not item, timestamp and price
    hourly_sum = hourly_sum.drop(columns=['register_id', 'currency', 'status'])

    return hourly_sum


In [139]:
def total_transaction_revenue(df_data: pd.DataFrame) -> pd.DataFrame:
    """Creates a pivot table displaying values for each status and total transaction revenue for each item."""

    pivot_table = pd.pivot_table(
        df_data,
        index='item',
        columns='status',
        values='price',
        aggfunc='sum',
        fill_value=0
    )

    pivot_table['total transaction revenue'] = pivot_table['SUCCESS'] - pivot_table['REFUND']

    return pivot_table


In [140]:
def refund_ratio_per_item(df_data: pd.DataFrame) -> pd.DataFrame:
    """Calculates the refund ratio per item and returns a dataframe with item and refund ratio."""

    sales_count_list = []
    refund_count_list = []
    item_list = []

    # Go through all items and count sales (SUCCESS) and refund (REFUND) per item
    for item, group in df_data.groupby('item'):
        status_count = group['status'].value_counts()
        sales = status_count.get('SUCCESS', 0)
        refund = status_count.get('REFUND', 0)
        sales_count_list.append(sales)
        refund_count_list.append(refund)
        item_list.append(item)

    # Convert the lists to numpy arrays
    sales_count_array = np.array(sales_count_list)
    refund_count_array = np.array(refund_count_list)
    item_array = np.array(item_list)

    # Calculate the refund ratio by calling function from task 3
    refund_ratio_array = compute_refund_ratio(sales_count_array, refund_count_array)

    # Make a new dataframe
    df_refund_ratio = pd.DataFrame({
        'item': item_array,
        'refund_ratio': refund_ratio_array
        })

    return df_refund_ratio
